
# Tutorial 1 — `hardware.py`

Our experiment has **detectors** (SPADs) that produce electrical pulses, and a **TimeTagger** that records *exactly when* each pulse arrived.

`hardware.py` is the **remote control** for:

1. **TimeTagger** — configure channels, measure count rates, release the device
2. **Rotation stages** — turn the half-wave plate (HWP) to change laser power

```
  SPAD detectors ──pulses──> TimeTagger ──> hardware.py configures it
  Half-wave plate ──USB──> Thorlabs stage ──> hardware.py rotates it
```

**Next tutorials:** `acquisition_tutorial.ipynb` (automated runs), `measurement_tutorial.ipynb` (live g² math on the tagger).


---
# Part A — Install the TimeTagger Python module

Our code uses `import TimeTagger as TT`. You need Swabian Instruments' software **and** the Python binding.


## Step 1 — Download the Time Tagger application

1. Go to [Swabian Instruments — Time Tagger downloads](https://www.swabianinstruments.com/time-tagger/downloads/)
2. Download **Time Tagger 2.x** for your OS (Windows installer or Linux package)
3. Install it (default location on Windows: `C:\Program Files\Swabian Instruments\Time Tagger\`)

This installs the **driver**, the **desktop GUI**, and (on Windows) the Python library.

## Step 2 — Python package (depends on your OS)

### Windows (typical lab PC)

The installer usually adds Python support automatically via the environment variable `PYTHONPATH`.

Default folder:
```
C:\Program Files\Swabian Instruments\Time Tagger\driver\python\
```

You should **not** need `pip install` on Windows if you installed the official app and use the same Python environment the installer configured.

### Linux

Either install the `.deb` / package from Swabian (puts files in system Python paths), **or** inside your virtualenv:

```bash
pip install Swabian-TimeTagger
```

(PyPI package name: `Swabian-TimeTagger`; requires Python ≥ 3.8 and numpy ≥ 1.25.)

### macOS

There is **no native macOS Time Tagger driver** for acquisition. You normally run this code on the **Windows or Linux lab PC** connected to the hardware. You can still read these tutorials on a Mac.


## Step 3 — Bootstrap

This finds the project folder and lets Python import our code from `src/`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "hardware.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


Project root: /Users/simon.wtmn/Desktop/Quantum_SHHG


## Step 4 — Verify the import

Run the next cell. Success looks like: `TimeTagger OK`. If it fails, see **Troubleshooting** at the end of this notebook.

In [ ]:
try:
    import TimeTagger as TT
    print("TimeTagger OK")
    print("  scanTimeTagger():", TT.scanTimeTagger())
except ImportError as e:
    print("TimeTagger NOT found:", e)
    print("\nFix:")
    print("  Windows: reinstall Time Tagger app; check PYTHONPATH")
    print("  Linux:   pip install Swabian-TimeTagger")
    print("  Docs:    https://www.swabianinstruments.com/static/documentation/TimeTagger/gettingStarted/installation.html")


TimeTagger NOT found: No module named 'TimeTagger'

Fix:
  Windows: reinstall Time Tagger app; check PYTHONPATH
  Linux:   pip install Swabian-TimeTagger
  Docs:    https://www.swabianinstruments.com/static/documentation/TimeTagger/gettingStarted/installation.html



## Lab vs laptop

| Flag | When to use |
|------|-------------|
| `LIVE_HARDWARE = False` | At home, on GitHub, learning the API (default) |
| `LIVE_HARDWARE = True` | On the **lab PC**, instruments plugged in |

In [3]:
LIVE_HARDWARE = False

# -------------------------
# edit these for YOUR bench 
# -------------------------
TT_SERIAL = ""                  # TimeTagger serial, or "" for first device
PRM1_SERIAL = "27264707"        # Thorlabs K-Cube serial (string)
ELL14_ADDRESS = 2               # Elliptec address (integer 0-9)
CHANNELS = [1, 2, 3, 4, 5, 6]   # list of channels to use


---
# Part B — Glossary

| Word | Meaning |
|------|----------------|
| **Channel** | One input on the TimeTagger (numbered 1, 2, 3…). One SPAD → one channel. |
| **Time tag** | One recorded event: "a pulse arrived at time T". |
| **Trigger level** | Voltage threshold (V). Pulse must cross this to count. |
| **Dead time** | Blind period (ps) after each tag on a channel. |
| **Input delay** | Shift timestamps (ps) to align cables / optics. |
| **Count rate** | How many tags per second on a channel. |
| **Resolution mode** | `"Standard"` or high-res modes — changes timing precision and which channels exist. |
| **Stage ID** | How we name a motor: **string** = PRM1 serial, **int 0–9** = ELL14 address. |



---
# Part C — `TimeTaggerDevice`

Think of it as a **wrapper object** around the real tagger. You create it, call `.connect()`, configure channels, then `.free()` when done.



## Typical workflow

```
1. dev = TimeTaggerDevice(serial="", resolution="Standard")
2. dev.connect()
3. dev.set_trigger_levels(...)
4. dev.set_input_delays(...)        # optional
5. dev.measure_countrate(...)       # optional check
6. ... acquisition code runs ...
7. dev.free()                       # or use: with TimeTaggerDevice() as dev:
```

### C.1 `TimeTaggerDevice.__init__(serial="", resolution="Standard", logger=None)`

**What it does:** Creates a Python object. **Does not** open USB yet.

| Parameter | Example | Meaning |
|-----------|---------|---------|
| `serial` | `""` or `"SN-12345"` | Which tagger. Empty = first found. |
| `resolution` | `"Standard"` | Must match a name in `TimeTagger.Resolution`. |
| `logger` | `None` | Optional - default logs to console. |


In [4]:
from src.hardware import TimeTaggerDevice

dev = TimeTaggerDevice(serial=TT_SERIAL, resolution="Standard")
print("Created. \nConnected...", dev.is_connected) 


Created. 
Connected... False



### C.2 `TimeTaggerDevice.scan()` — list devices on USB

Static method: call as `TimeTaggerDevice.scan()` without creating an object first.


In [5]:

if LIVE_HARDWARE:
    print("Serial numbers:", TimeTaggerDevice.scan())
else:
    print("[offline] Would return list of serial strings, e.g. ['TT-1234567890']")


[offline] Would return list of serial strings, e.g. ['TT-1234567890']



### C.3 `connect()` and `is_connected`

- `connect()` opens the device, fills `dev.model` and `dev.serial`
- `is_connected` is a property (`True` / `False`)
- Raises `RuntimeError` if another program holds the tagger (close Swabian GUI first!)


In [6]:

if LIVE_HARDWARE:
    dev.connect()
    print(f"Model: {dev.model}, serial: {dev.serial}, connected: {dev.is_connected}")
else:
    print("[offline] connect() opens USB and checks model is supported.")


[offline] connect() opens USB and checks model is supported.



### C.4 `available_channels(rising=True)` and `validate_channels(channels)`

Before measuring, check your channel list is valid **for this resolution mode**.

Example: you want channels `[1,2,3,4,5,6]` for the LOA setup.


In [7]:

if LIVE_HARDWARE and dev.is_connected:
    print("Available:", dev.available_channels())
    print("Our list OK?", dev.validate_channels(CHANNELS))
else:
    print("[offline] validate_channels([1..6]) -> True if all exist on device.")


[offline] validate_channels([1..6]) -> True if all exist on device.



### C.5 Channel settings (all need `connect()` first)

#### `set_trigger_levels(channels, levels)` — threshold in **Volts**

- One number → same voltage on all channels: `set_trigger_levels(CHANNELS, 0.5)`
- List → one per channel: `set_trigger_levels([1,2], [0.4, 0.6])`

#### `set_dead_times(channels, dead_times_ps)` — dead time in **picoseconds**

Same length as `channels`. Use `0` for no extra dead time.

#### `set_input_delays(channels, delays_ps)` — delay in **picoseconds**

Aligns paths in software. Lab example:
`[0, -1000, 20000, 16500, 17800, 17300]`

#### `set_conditional_filter(trigger, filtered, hardware_delay_compensation=True)`

Advanced: only keep tags on `filtered` channels after a tag on `trigger`. Empty lists = does nothing.

#### `set_test_signal(channels, enabled=True)`

Internal test pulses — useful to check wiring without laser light.


In [8]:

if LIVE_HARDWARE and dev.is_connected:
    dev.set_trigger_levels(CHANNELS, 0.5)
    dev.set_dead_times(CHANNELS, [0]*6)
    dev.set_input_delays(CHANNELS, [0, -1000, 20000, 16500, 17800, 17300])
    print("Channel settings applied.")
else:
    print("[offline] See markdown above for each method.")


[offline] See markdown above for each method.



### C.6 `measure_countrate(channels, duration_s=5.0)`

**Purpose:** Quick "how many photons per second?" check (background vs signal).

**Returns:** `{1: 12500.0, 2: 9800.0, ...}`  (channel → counts/s)

**Lab tip:** Run once with beam blocked, once unblocked, before a long acquisition.


In [9]:

if LIVE_HARDWARE and dev.is_connected:
    rates = dev.measure_countrate(CHANNELS, duration_s=3)
    for ch, r in sorted(rates.items()):
        print(f"  channel {ch}: {r:,.0f} counts/s")
else:
    print("[offline] Returns dict channel -> counts per second.")


[offline] Returns dict channel -> counts per second.



### C.7 `free()` and the `with` block

**Always release the tagger** when finished, otherwise the next script gets "device busy".

Recommended pattern:
```python
with TimeTaggerDevice(resolution="Standard") as dev:
    dev.set_trigger_levels([1, 2], 0.5)
    print(dev.measure_countrate([1, 2], 2.0))
# dev.free() called automatically here
```


In [10]:

if LIVE_HARDWARE and dev.is_connected:
    dev.free()
    print("Freed. is_connected =", dev.is_connected)
else:
    print("[offline] Always call free() or use `with TimeTaggerDevice(...) as dev:`")


[offline] Always call free() or use `with TimeTaggerDevice(...) as dev:`



---
# Part D — Rotation stages

We rotate a **half-wave plate** to change power. Two hardware types are supported:

| Class | Hardware | ID type | Library |
|-------|----------|---------|---------|
| `PRM1Stage` | Thorlabs PRM1 on K-Cube | `"27264707"` (string) | Kinesis + pythonnet |
| `ELL14Stage` | Thorlabs ELL14 | `2` (int 0–9) | elliptec + pyserial |

`RotationStage` itself is **abstract** — you never instantiate it directly.

Shared method on all stages: **`move_to(angle_deg, extra_delay_s=0.1)`** — rotates and waits until done. Angle wraps to 0–360°.



### D.1 `PRM1Stage`

**Install (Windows):** Thorlabs Kinesis + `conda install pythonnet`

| Method | What it does |
|--------|----------------|
| `PRM1Stage(serial)` | Store serial number |
| `PRM1Stage.list_available()` | Scan USB for PRM1 serials (static) |
| `.connect()` | Open motor |
| `.move_to(angle)` | Rotate (inherited) |
| `.position()` | Current angle (°) or `None` |
| `.home()` | Find reference position |
| `.disconnect()` | Release motor |


In [11]:

from src.hardware import PRM1Stage

print("PRM1 public methods:", [m for m in dir(PRM1Stage) if not m.startswith("_") and callable(getattr(PRM1Stage,m))])

if LIVE_HARDWARE:
    print("Found:", PRM1Stage.list_available())
    stage = PRM1Stage(PRM1_SERIAL)
    stage.connect()
    print("Position:", stage.position(), "deg")
    # stage.move_to(45)  # uncomment to move — physical motion!
    stage.disconnect()
else:
    print("[offline] Example: PRM1Stage('27264707').connect().move_to(50).disconnect()")


PRM1 public methods: ['connect', 'disconnect', 'home', 'list_available', 'move_to', 'position']
[offline] Example: PRM1Stage('27264707').connect().move_to(50).disconnect()



### D.2 `ELL14Stage`

**Install:** `pip install elliptec pyserial`

Several ELL14 mounts share **one** serial bus. `ELL14Stage.close_controller()` closes that bus when you're completely done.

| Method | What it does |
|--------|----------------|
| `ELL14Stage(address, port=None)` | Address 0–9; optional COM port |
| `.connect()` | Bind to shared controller |
| `.move_to(angle)` | Rotate |
| `.position()` / `.home()` / `.disconnect()` | As PRM1 |
| `ELL14Stage.close_controller()` | Close serial port (class method) |


In [12]:

from src.hardware import ELL14Stage

if LIVE_HARDWARE:
    ell = ELL14Stage(ELL14_ADDRESS, port=None)
    ell.connect()
    print("Position:", ell.position())
    ell.disconnect()
    ELL14Stage.close_controller()
else:
    print("[offline] Pass port='COM3' if auto-detect fails.")


[offline] Pass port='COM3' if auto-detect fails.



### D.3 `RotationStageController` — the easy way (used in acquisition)

One object manages **several** stages. ID type picks the driver automatically.

```python
from src.hardware import RotationStageController

with RotationStageController(["27264707"]) as stages:
    stages.move_to(55.0, "27264707")   # degrees
    print(stages.position())
```

| Method | `stage_id=None` | `stage_id="27264707"` |
|--------|-----------------|------------------------|
| `move_to(angle, stage_id, extra_delay_s=0.1)` | first stage only | that stage |
| `position(stage_id)` | dict of all | one float |
| `home(stage_id)` | home all | home one |
| `disconnect_all()` | release everything | |


In [13]:

from src.hardware import RotationStageController

if LIVE_HARDWARE:
    with RotationStageController([PRM1_SERIAL]) as stages:
        print("Positions:", stages.position())
else:
    print("[offline] RotationStageController([PRM1_SERIAL]).connect().move_to(50, PRM1_SERIAL)")


[offline] RotationStageController([PRM1_SERIAL]).connect().move_to(50, PRM1_SERIAL)



---
# Part E — Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError: TimeTagger` | Install Swabian app (Windows) or `pip install Swabian-TimeTagger` (Linux venv) |
| `No Time Tagger found` | Close Time Tagger GUI; unplug/replug USB; call `.free()` |
| `call connect() first` | Run `.connect()` before config methods |
| `Kinesis DLLs not found` | Install Thorlabs Kinesis; set env `KINESIS_DLL_PATH` |
| Stage doesn't move | `RUN_MOTION` / check `stage_id`; verify USB |

**Official docs:** [Time Tagger installation](https://www.swabianinstruments.com/static/documentation/TimeTagger/gettingStarted/installation.html)

**Next:** open `acquisition_tutorial.ipynb` to see how `hardware.py` is used in a full measurement.
